# Checkpoint C1: Chunker.py — Tách tài liệu thành chunks

Đây là kết quả chạy chunker.py. Notebook này load các tài liệu chính sách HR,
tách thành chunks theo heading, và hiển thị kết quả.

**Cách sử dụng:** Chạy tất cả cells để xem chunker hoạt động.

In [29]:
# Cài đặt các thư viện cần thiết cho RAG và Vector Search
print("Đang kiểm tra và cài đặt chromadb, sentence-transformers...")
!pip install -q chromadb sentence-transformers jsonschema
print("✓ Cài đặt hoàn tất!")

Đang kiểm tra và cài đặt chromadb, sentence-transformers...
✓ Cài đặt hoàn tất!


In [30]:
import sys
import os
import json
import shutil
from pathlib import Path

TEMPLATE_SKILL_DIR = Path("../../templates/skills/hr-policy-qa-skill")
OUTPUT_SKILL_DIR = Path("../../outputs/skills/hr-policy-qa-skill")

# Tạo thư mục scripts/ nếu chưa tồn tại
(OUTPUT_SKILL_DIR / "scripts").mkdir(parents=True, exist_ok=True)

# Sao chép chunker.py sang outputs
src_chunker = TEMPLATE_SKILL_DIR / "scripts" / "chunker.py"
dst_chunker = OUTPUT_SKILL_DIR / "scripts" / "chunker.py"
if src_chunker.exists():
    shutil.copy(src_chunker, dst_chunker)
    print(f"✓ Đã sao chép chunker.py sang: {dst_chunker}")

SCRIPTS_DIR = (OUTPUT_SKILL_DIR / "scripts").resolve()
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

print(f"Scripts dir: {SCRIPTS_DIR}")
print(f"Exists: {SCRIPTS_DIR.exists()}")


✓ Đã sao chép chunker.py sang: ../../outputs/skills/hr-policy-qa-skill/scripts/chunker.py
Scripts dir: /Users/shimazu/Documents/9. active/course_viettel_network/vtn-ai-builders-bootcamp-2026/03-practice/day-02/session-04-knowledge-base-rag-hr-policy-qa/outputs/skills/hr-policy-qa-skill/scripts
Exists: True


In [31]:
# Thử import chunker module
try:
    from chunker import (
        load_policy_files,
        chunk_markdown_by_heading,
        generate_metadata,
        run_pipeline,
    )
    HAS_CHUNKER = True
    print("Đã import chunker module thành công!")
    print("  - load_policy_files")
    print("  - chunk_markdown_by_heading")
    print("  - generate_metadata")
    print("  - run_pipeline")
except ImportError as e:
    HAS_CHUNKER = False
    print(f"Không thể import chunker module: {e}")
    print("Sẽ định nghĩa các hàm inline...")
    
    # Inline định nghĩa các hàm chunker (fallback)
    import re
    import uuid
    from typing import Any

    def _parse_frontmatter(text):
        match = re.match(r"^---\s*\n(.*?)\n---\s*\n", text, re.DOTALL)
        if not match:
            return {}, text
        frontmatter_str = match.group(1)
        body = text[match.end():]
        metadata = {}
        for line in frontmatter_str.strip().splitlines():
            if ":" in line:
                key, _, value = line.partition(":")
                metadata[key.strip()] = value.strip().strip('"')
        return metadata, body

    def load_policy_files(kb_dir):
        kb_path = Path(kb_dir)
        if not kb_path.exists():
            return []
        documents = []
        for md_file in sorted(kb_path.glob("*.md")):
            raw = md_file.read_text(encoding="utf-8")
            metadata, body = _parse_frontmatter(raw)
            doc_id = metadata.get("doc_id", md_file.stem)
            documents.append({
                "filename": md_file.name,
                "doc_id": doc_id,
                "metadata": metadata,
                "content": body,
            })
        return documents

    def chunk_markdown_by_heading(content, max_words=500, overlap_words=50):
        parts = re.split(r"(?=^##\s+)", content, flags=re.MULTILINE)
        chunks = []
        chunk_index = 0
        for part in parts:
            part = part.strip()
            if not part:
                continue
            heading_match = re.match(r"^##\s+(.+?)(?:\n|$)", part)
            if heading_match:
                section_name = heading_match.group(1).strip()
                body = part[heading_match.end():].strip()
            else:
                section_name = "Tổng quan"
                body = part.strip()
            if not body:
                continue
            words = body.split()
            if len(words) <= max_words:
                chunks.append({"section": section_name, "content": body, "chunk_index": chunk_index})
                chunk_index += 1
            else:
                start = 0
                while start < len(words):
                    end = start + max_words
                    sub_text = " ".join(words[start:end])
                    chunks.append({"section": section_name, "content": sub_text, "chunk_index": chunk_index})
                    chunk_index += 1
                    start += max_words - overlap_words
        return chunks

    print("Đã định nghĩa inline functions thành công!")

Đã import chunker module thành công!
  - load_policy_files
  - chunk_markdown_by_heading
  - generate_metadata
  - run_pipeline


In [32]:
# Load các tài liệu chính sách HR từ thư mục outputs của học viên
KB_DIR = Path("../../outputs/skills/hr-policy-qa-skill/kb/hr-policies")

documents = load_policy_files(str(KB_DIR))

print(f"Đã load {len(documents)} tài liệu chính sách HR từ outputs:")
print("=" * 60)
for doc in documents:
    doc_id = doc.get("doc_id", "?")
    filename = doc.get("filename", "?")
    content_len = len(doc.get("content", ""))
    word_count = len(doc.get("content", "").split())
    version = doc.get("metadata", {}).get("version", "?")
    print(f"  {doc_id} ({version}) — {filename}")
    print(f"    {content_len} ký tự, {word_count} từ")
    print()

if not documents:
    print("[CẢNH BÁO] Không tìm thấy tài liệu. Kiểm tra đường dẫn:")
    print(f"  {KB_DIR.resolve()}")
    print(f"  Exists: {KB_DIR.exists()}")


[chunker] Loaded 4 policy file(s) from ../../outputs/skills/hr-policy-qa-skill/kb/hr-policies
Đã load 4 tài liệu chính sách HR từ outputs:
  POL-ALLOW-001 (v1.3) — policy-allowance.md
    2233 ký tự, 513 từ

  POL-LEAVE-001 (v2.1) — policy-leave.md
    2444 ký tự, 563 từ

  POL-SENIOR-001 (v1.0) — policy-seniority.md
    2077 ký tự, 500 từ

  POL-TRAIN-001 (v1.1) — policy-training.md
    3052 ký tự, 674 từ



In [33]:
# Chạy chunk_markdown_by_heading() trên policy-leave.md
leave_doc = None
for doc in documents:
    if "leave" in doc.get("filename", "").lower():
        leave_doc = doc
        break

if leave_doc:
    print(f"Đang chunking: {leave_doc['filename']} ({leave_doc['doc_id']})")
    print("=" * 60)
    
    chunks = chunk_markdown_by_heading(leave_doc["content"])
    
    print(f"Tổng số chunks: {len(chunks)}\n")
    
    for i, chunk in enumerate(chunks):
        section = chunk["section"]
        content = chunk["content"]
        word_count = len(content.split())
        
        print(f"--- Chunk {i}: {section} ({word_count} từ) ---")
        # Hiển thị 3 dòng đầu
        preview_lines = content.split("\n")[:3]
        for line in preview_lines:
            print(f"  {line[:80]}")
        if len(preview_lines) < content.count("\n") + 1:
            print(f"  ... (còn thêm {content.count(chr(10)) - 2} dòng)")
        print()
else:
    print("[LỖI] Không tìm thấy policy-leave.md trong danh sách tài liệu")

Đang chunking: policy-leave.md (POL-LEAVE-001)
Tổng số chunks: 5

--- Chunk 0: Tổng quan (22 từ) ---
  # Chính sách nghỉ phép năm, nghỉ ốm và nghỉ thai sản
  
  *(Tài liệu mô phỏng — không phải chính sách thật)*

--- Chunk 1: 1. Nghỉ phép năm (189 từ) ---
  ### 1.1 Số ngày phép
  
  Nhân viên chính thức được hưởng ngày phép năm theo thâm niên:
  ... (còn thêm 21 dòng)

--- Chunk 2: 2. Nghỉ ốm (130 từ) ---
  ### 2.1 Điều kiện
  
  - Nghỉ ốm hưởng nguyên lương trong 30 ngày đầu mỗi năm.
  ... (còn thêm 13 dòng)

--- Chunk 3: 3. Nghỉ thai sản (121 từ) ---
  ### 3.1 Thời gian nghỉ
  
  - Thai sản thông thường: 6 tháng.
  ... (còn thêm 13 dòng)

--- Chunk 4: 4. Các loại nghỉ khác (81 từ) ---
  | Loại nghỉ | Số ngày | Lương |
  | --- | ---: | --- |
  | Nghỉ việc riêng (kết hôn, tang) | 3 ngày | 100% |
  ... (còn thêm 4 dòng)



In [34]:
# Chạy chunking trên tất cả 4 file, hiển thị bảng tổng hợp
print("BẢNG TỔNG HỢP CHUNKING")
print("=" * 65)
print(f"{'doc_id':<18} {'Tên file':<22} {'Số chunks':<10} {'Từ TB':<10}")
print("-" * 65)

all_chunks_all_docs = []

for doc in documents:
    doc_id = doc.get("doc_id", "?")
    filename = doc.get("filename", "?")
    chunks = chunk_markdown_by_heading(doc["content"])
    
    chunk_word_counts = [len(c["content"].split()) for c in chunks]
    avg_words = sum(chunk_word_counts) / len(chunk_word_counts) if chunk_word_counts else 0
    
    print(f"{doc_id:<18} {filename:<22} {len(chunks):<10} {avg_words:<10.0f}")
    
    for c in chunks:
        c["doc_id"] = doc_id
    all_chunks_all_docs.extend(chunks)

print("-" * 65)
print(f"{'TỔNG':<18} {'':<22} {len(all_chunks_all_docs):<10}")

BẢNG TỔNG HỢP CHUNKING
doc_id             Tên file               Số chunks  Từ TB     
-----------------------------------------------------------------
POL-ALLOW-001      policy-allowance.md    4          124       
POL-LEAVE-001      policy-leave.md        5          109       
POL-SENIOR-001     policy-seniority.md    5          95        
POL-TRAIN-001      policy-training.md     6          107       
-----------------------------------------------------------------
TỔNG                                      20        


In [35]:
# Thử ChromaDB — nếu có sẵn thì populate, không thì hiển thị fallback
try:
    import chromadb
    HAS_CHROMA = True
    print("ChromaDB đã sẵn sàng!")
except ImportError:
    HAS_CHROMA = False
    print("ChromaDB chưa cài đặt.")
    print("Cài đặt: pip install chromadb")
    print()
    print("Fallback: Chunks đã được tách và sẵn sàng để sử dụng.")
    print("Khi ChromaDB được cài đặt, bạn có thể:")
    print("  1. Chạy chunker.py --chroma để populate ChromaDB")
    print("  2. Sử dụng retriever.py để truy xuất vector search")

if HAS_CHROMA:
    # Tạo ChromaDB collection và thêm chunks
    client = chromadb.Client()
    collection = client.get_or_create_collection(
        name="hr_policies_demo",
        metadata={"description": "HR Policy chunks - Checkpoint C1 demo"},
    )
    
    # Chuẩn bị dữ liệu
    texts = [c["content"] for c in all_chunks_all_docs]
    ids = [f"chunk-{i}" for i in range(len(all_chunks_all_docs))]
    metas = [{"doc_id": c.get("doc_id", "?"), "section": c.get("section", "?")} for c in all_chunks_all_docs]
    
    collection.add(documents=texts, ids=ids, metadatas=metas)
    
    print(f"\nChromaDB Collection: {collection.name}")
    print(f"  Số chunks: {collection.count()}")
    print(f"  Metadata fields: doc_id, section")
    print("\nCollection sẵn sàng cho vector search!")

ChromaDB đã sẵn sàng!


/Users/shimazu/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:15<00:00, 5.45MiB/s]



ChromaDB Collection: hr_policies_demo
  Số chunks: 20
  Metadata fields: doc_id, section

Collection sẵn sàng cho vector search!


### Chạy Script Chunker từ Command Line

Bây giờ chúng ta sẽ chạy script `chunker.py` từ dòng lệnh để tự động tạo ra file `kb/chunks.json` chứa các chunks của chính sách nhân sự.

In [36]:
# Chạy chunker.py qua dòng lệnh CLI để xuất chunks.json
!python ../../outputs/skills/hr-policy-qa-skill/scripts/chunker.py --kb-dir ../../outputs/skills/hr-policy-qa-skill/kb/hr-policies --output ../../outputs/skills/hr-policy-qa-skill/kb/chunks.json

# Kiểm tra xem chunks.json đã được tạo chưa
chunks_json_path = Path("../../outputs/skills/hr-policy-qa-skill/kb/chunks.json")
print(f"\nchunks.json exists: {chunks_json_path.exists()}")
if chunks_json_path.exists():
    with open(chunks_json_path, "r", encoding="utf-8") as f:
        chunks_data = json.load(f)
    print(f"Tổng số chunks được ghi nhận trong file: {len(chunks_data)}")


[chunker] Loaded 4 policy file(s) from ../../outputs/skills/hr-policy-qa-skill/kb/hr-policies
[chunker] Generated 20 chunk(s) total
[chunker] Saved chunks to ../../outputs/skills/hr-policy-qa-skill/kb/chunks.json

chunks.json exists: True
Tổng số chunks được ghi nhận trong file: 20


In [37]:
# Tổng kết cuối cùng
total_chunks = len(all_chunks_all_docs)
all_words = [len(c["content"].split()) for c in all_chunks_all_docs]
total_words = sum(all_words)
avg_chunk_size = total_words / total_chunks if total_chunks > 0 else 0

print("=" * 60)
print("TỔNG KẾT CHECKPOINT C1")
print("=" * 60)
print(f"Số tài liệu đã xử lý:  {len(documents)}")
print(f"Tổng số chunks tạo ra: {total_chunks}")
print(f"Trung bình từ/chunk:   {avg_chunk_size:.0f}")
print(f"Tổng số từ đã xử lý:   {total_words}")
print(f"ChromaDB:              {'Có' if HAS_CHROMA else 'Chưa cài đặt (fallback)'}")
print()
print("Phân tích chi tiết:")
for doc in documents:
    chunks = chunk_markdown_by_heading(doc["content"])
    sections = set(c["section"] for c in chunks)
    print(f"  {doc['doc_id']}: {len(chunks)} chunks, {len(sections)} sections")
print()
print("Checkpoint C1 hoàn thành! Chunker đã chia nhỏ các tài liệu chính sách thành chunks thành công tại outputs/skills/hr-policy-qa-skill/kb/chunks.json và nạp vào ChromaDB.")
print("Bạn đã hiểu cách chunker.py tách tài liệu thành chunks.")
print("Tiếp tục lab để xây dựng retriever.py và evaluator.py.")

TỔNG KẾT CHECKPOINT C1
Số tài liệu đã xử lý:  4
Tổng số chunks tạo ra: 20
Trung bình từ/chunk:   108
Tổng số từ đã xử lý:   2154
ChromaDB:              Có

Phân tích chi tiết:
  POL-ALLOW-001: 4 chunks, 4 sections
  POL-LEAVE-001: 5 chunks, 5 sections
  POL-SENIOR-001: 5 chunks, 5 sections
  POL-TRAIN-001: 6 chunks, 6 sections

Checkpoint C1 hoàn thành! Chunker đã chia nhỏ các tài liệu chính sách thành chunks thành công tại outputs/skills/hr-policy-qa-skill/kb/chunks.json và nạp vào ChromaDB.
Bạn đã hiểu cách chunker.py tách tài liệu thành chunks.
Tiếp tục lab để xây dựng retriever.py và evaluator.py.
